In [1]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_ZHOU2016_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [2]:
from pipelines import mi_pipelines, ood_pipelines
from processing import *

/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/optuna/integration/sklearn.py:14: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  optuna_warn(f"{msg} Use `optuna_integration.sklearn` instead.", FutureWarning)
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [3]:
experiment = "cross-session_7-channels-rest"

COMMON_CHANNELS = ["FC3", "FCz", "FC4", "C3", "C4", "CP3", "CP4"]

datasets = [ "Zhou2016", "BNCI2014_001", "TTK", "Zhou2020", "Wang2026_45", "Stieger2021_fix" ] # 4 9 6 20, 39, 62


MI_METHODS = ['ACM+TS+SVM', 'FBCSP+ElasticNet', 'ShallowConvNet', 'EEGNet', "Microstates+GB", "VAR+shLDA", "Log-bandpower+shLDA" ]
OOD_METHODS = ['ACM+TS+OCSVM', 'FBCSP+IsolationForest', 'ShallowConvNet+kNN', 'EEGNet+TempDens', "Log-bandpower+Mahalanobis", "Microstates+LOF", "VAR+Mahalanobis" ]


In [4]:
for dataset in datasets:
    print( dataset )
    subjects = get_subjects(dataset) 

    for subject in subjects:
        data = None
        
        for method, pipeline in mi_pipelines.items():
            if method not in MI_METHODS:
                continue
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": "mi", "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, duration=3.0, print_count=True)
            train_data, val_data, test_data = data
            result["MI_BA"], result["MI_AUROC"] = MI_metrics(train_data, test_data, pipeline)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['MI_BA']:.3f} {result['MI_AUROC']:.3f}")

        data = None
        for method, pipeline in ood_pipelines.items():
            if method not in OOD_METHODS:
                continue
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": "ood", "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, ood_classes=("rest",), duration=3.0, print_count=True )
                train_data, val_data, test_data = data

            model = train(train_data, pipeline)

            result["OOD_AUROC"] = OOD_AUROC(test_data, model)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['OOD_AUROC']:.3f}")

    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_BA")
    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_AUROC")
    display_summary_by_subject(dataset, experiment, experiment_type="ood", metric="OOD_AUROC")

Zhou2016


,MI_BA,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
EEGNet,0.847000,4,0.810000,0.778000,0.910000,0.890000
ShallowConvNet,0.784000,4,0.700000,0.678000,0.890000,0.870000
FBCSP+ElasticNet,0.774000,4,0.690000,0.644000,0.890000,0.870000
ACM+TS+SVM,0.738000,4,0.610000,0.633000,0.790000,0.920000
Microstates+GB,0.717000,4,0.530000,0.589000,0.840000,0.910000
Log-bandpower+shLDA,0.697000,4,0.640000,0.689000,0.810000,0.650000
VAR+shLDA,0.641000,4,0.680000,0.633000,0.560000,0.690000


,MI_AUROC,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
EEGNet,0.921000,4,0.893000,0.841000,0.973000,0.978000
ShallowConvNet,0.891000,4,0.875000,0.766000,0.978000,0.945000
FBCSP+ElasticNet,0.875000,4,0.870000,0.746000,0.947000,0.936000
ACM+TS+SVM,0.858000,4,0.810000,0.719000,0.927000,0.978000
Log-bandpower+shLDA,0.805000,4,0.762000,0.755000,0.914000,0.787000
Microstates+GB,0.780000,4,0.590000,0.664000,0.901000,0.966000
VAR+shLDA,0.681000,4,0.677000,0.731000,0.563000,0.753000


,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
EEGNet+TempDens,0.771000,4,0.902000,0.871000,0.724000,0.588000
ShallowConvNet+kNN,0.728000,4,0.716000,0.907000,0.654000,0.634000
ACM+TS+OCSVM,0.714000,4,0.823000,0.868000,0.598000,0.566000
FBCSP+IsolationForest,0.712000,4,0.848000,0.835000,0.727000,0.436000
Log-bandpower+Mahalanobis,0.644000,4,0.758000,0.793000,0.627000,0.397000
VAR+Mahalanobis,0.641000,4,0.702000,0.801000,0.568000,0.494000
Microstates+LOF,0.620000,4,0.626000,0.791000,0.544000,0.518000


BNCI2014_001


,MI_BA,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09
method,,,,,,,,,,,
EEGNet,0.781000,9,0.889000,0.576000,0.958000,0.778000,0.576000,0.743000,0.632000,0.972000,0.903000
ACM+TS+SVM,0.752000,9,0.826000,0.590000,0.938000,0.722000,0.639000,0.722000,0.576000,0.986000,0.764000
FBCSP+ElasticNet,0.750000,9,0.840000,0.604000,0.958000,0.590000,0.660000,0.681000,0.688000,0.958000,0.771000
ShallowConvNet,0.731000,9,0.819000,0.514000,0.896000,0.667000,0.569000,0.715000,0.597000,0.944000,0.854000
Microstates+GB,0.677000,9,0.785000,0.493000,0.854000,0.597000,0.556000,0.639000,0.542000,0.917000,0.715000
Log-bandpower+shLDA,0.653000,9,0.653000,0.500000,0.812000,0.576000,0.549000,0.625000,0.590000,0.785000,0.785000
VAR+shLDA,0.649000,9,0.646000,0.535000,0.840000,0.549000,0.597000,0.646000,0.569000,0.715000,0.743000


... (14 outputs left out)

Stieger2021_fix


,MI_BA,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20,sub-21,sub-22,sub-23,sub-24,sub-25,sub-26,sub-27,sub-28,sub-29,sub-30,sub-31,sub-32,sub-33,sub-34,sub-35,sub-36,sub-37,sub-38,sub-39,sub-40,sub-41,sub-42,sub-43,sub-44,sub-45,sub-46,sub-47,sub-48,sub-49,sub-50,sub-51,sub-52,sub-53,sub-54,sub-55,sub-56,sub-57,sub-58,sub-59,sub-60,sub-61,sub-62
method,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FBCSP+ElasticNet,0.624000,62,0.796000,0.583000,0.535000,0.833000,0.744000,0.607000,0.488000,0.629000,0.702000,0.558000,0.534000,0.680000,0.542000,0.544000,0.544000,0.576000,0.503000,0.565000,0.750000,0.585000,0.553000,0.589000,0.530000,0.808000,0.660000,0.592000,0.562000,0.959000,0.827000,0.799000,0.491000,0.501000,0.569000,0.500000,0.712000,0.626000,0.802000,0.783000,0.763000,0.606000,0.545000,0.477000,0.561000,0.432000,0.697000,0.707000,0.548000,0.481000,0.686000,0.887000,0.500000,0.648000,0.656000,0.668000,0.493000,0.677000,0.814000,0.555000,0.500000,0.450000,0.578000,0.620000
EEGNet,0.615000,62,0.768000,0.715000,0.418000,0.750000,0.706000,0.641000,0.565000,0.803000,0.696000,0.533000,0.527000,0.570000,0.533000,0.554000,0.392000,0.538000,0.496000,0.566000,0.778000,0.800000,0.626000,0.500000,0.692000,0.712000,0.706000,0.655000,0.480000,0.932000,0.783000,0.813000,0.544000,0.605000,0.473000,0.513000,0.740000,0.662000,0.846000,0.636000,0.625000,0.558000,0.510000,0.456000,0.559000,0.514000,0.542000,0.778000,0.557000,0.631000,0.442000,0.887000,0.531000,0.624000,0.566000,0.496000,0.511000,0.566000,0.804000,0.463000,0.532000,0.636000,0.602000,0.458000
ACM+TS+SVM,0.608000,62,0.690000,0.640000,0.493000,0.762000,0.617000,0.574000,0.500000,0.549000,0.692000,0.563000,0.537000,0.610000,0.556000,0.560000,0.766000,0.652000,0.578000,0.476000,0.516000,0.659000,0.577000,0.532000,0.530000,0.705000,0.686000,0.690000,0.609000,0.667000,0.851000,0.627000,0.424000,0.558000,0.610000,0.511000,0.577000,0.685000,0.833000,0.824000,0.833000,0.563000,0.511000,0.560000,0.571000,0.489000,0.559000,0.700000,0.519000,0.567000,0.523000,0.839000,0.463000,0.706000,0.720000,0.625000,0.580000,0.500000,0.799000,0.566000,0.500000,0.464000,0.545000,0.533000
ShallowConvNet,0.603000,62,0.746000,0.737000,0.508000,0.619000,0.542000,0.560000,0.452000,0.628000,0.590000,0.521000,0.583000,0.516000,0.549000,0.598000,0.571000,0.605000,0.518000,0.483000,0.734000,0.618000,0.604000,0.453000,0.612000,0.664000,0.510000,0.838000,0.541000,0.854000,0.709000,0.881000,0.481000,0.587000,0.473000,0.533000,0.600000,0.531000,0.720000,0.718000,0.651000,0.551000,0.457000,0.487000,0.620000,0.543000,0.506000,0.726000,0.567000,0.513000,0.607000,0.943000,0.520000,0.766000,0.582000,0.574000,0.575000,0.553000,0.656000,0.597000,0.561000,0.594000,0.631000,0.621000
VAR+shLDA,0.602000,62,0.535000,0.720000,0.547000,0.669000,0.567000,0.654000,0.548000,0.690000,0.738000,0.597000,0.488000,0.551000,0.556000,0.552000,0.474000,0.623000,0.506000,0.439000,0.767000,0.648000,0.588000,0.436000,0.608000,0.705000,0.597000,0.738000,0.563000,0.737000,0.631000,0.672000,0.483000,0.543000,0.676000,0.524000,0.628000,0.575000,0.771000,0.833000,0.490000,0.568000,0.538000,0.479000,0.616000,0.550000,0.603000,0.614000,0.673000,0.490000,0.650000,0.854000,0.520000,0.751000,0.660000,0.631000,0.469000,0.514000,0.741000,0.487000,0.594000,0.445000,0.578000,0.616000
Log-bandpower+shLDA,0.591000,62,0.655000,0.673000,0.567000,0.700000,0.506000,0.637000,0.491000,0.672000,0.630000,0.564000,0.520000,0.467000,0.451000,0.499000,0.545000,0.553000,0.613000,0.479000,0.748000,0.602000,0.617000,0.518000,0.608000,0.740000,0.526000,0.912000,0.524000,0.761000,0.551000,0.627000,0.481000,0.532000,0.547000,0.500000,0.689000,0.599000,0.697000,0.705000,0.596000,0.598000,0.510000,0.511000,0.555000,0.508000,0.532000,0.633000,0.539000,0.509000,0.580000,0.871000,0.541000,0.672000,0.592000,0.636000,0.525000,0.547000,0.621000,0.555000,0.638000,0.521000,0.61

,MI_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20,sub-21,sub-22,sub-23,sub-24,sub-25,sub-26,sub-27,sub-28,sub-29,sub-30,sub-31,sub-32,sub-33,sub-34,sub-35,sub-36,sub-37,sub-38,sub-39,sub-40,sub-41,sub-42,sub-43,sub-44,sub-45,sub-46,sub-47,sub-48,sub-49,sub-50,sub-51,sub-52,sub-53,sub-54,sub-55,sub-56,sub-57,sub-58,sub-59,sub-60,sub-61,sub-62
method,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FBCSP+ElasticNet,0.715000,62,0.919000,0.944000,0.558000,0.817000,0.892000,0.683000,0.496000,0.789000,0.863000,0.646000,0.592000,0.644000,0.567000,0.586000,0.680000,0.625000,0.595000,0.501000,0.941000,0.631000,0.669000,0.558000,0.784000,0.860000,0.783000,0.987000,0.654000,0.976000,0.914000,0.974000,0.438000,0.505000,0.610000,0.646000,0.765000,0.721000,0.984000,0.966000,0.853000,0.746000,0.554000,0.536000,0.572000,0.460000,0.775000,0.827000,0.655000,0.466000,0.801000,0.990000,0.559000,0.719000,0.814000,0.762000,0.444000,0.764000,0.891000,0.574000,0.899000,0.591000,0.617000,0.674000
ACM+TS+SVM,0.688000,62,0.793000,0.718000,0.563000,0.913000,0.667000,0.662000,0.526000,0.672000,0.877000,0.672000,0.542000,0.665000,0.575000,0.633000,0.804000,0.715000,0.597000,0.493000,0.918000,0.664000,0.738000,0.641000,0.610000,0.803000,0.779000,0.953000,0.655000,0.855000,0.900000,0.903000,0.432000,0.541000,0.655000,0.454000,0.655000,0.754000,0.921000,0.908000,0.987000,0.635000,0.503000,0.535000,0.705000,0.470000,0.691000,0.767000,0.535000,0.551000,0.706000,0.980000,0.500000,0.774000,0.780000,0.692000,0.674000,0.690000,0.914000,0.612000,0.372000,0.522000,0.592000,0.617000
EEGNet,0.678000,62,0.870000,0.926000,0.393000,0.847000,0.739000,0.700000,0.526000,0.897000,0.806000,0.582000,0.537000,0.603000,0.511000,0.561000,0.386000,0.597000,0.492000,0.572000,0.880000,0.883000,0.736000,0.493000,0.885000,0.823000,0.730000,0.955000,0.501000,0.998000,0.892000,0.941000,0.561000,0.602000,0.544000,0.683000,0.779000,0.688000,0.919000,0.818000,0.949000,0.677000,0.531000,0.443000,0.566000,0.564000,0.602000,0.852000,0.539000,0.690000,0.513000,0.983000,0.481000,0.653000,0.584000,0.532000,0.531000,0.677000,0.915000,0.470000,0.599000,0.637000,0.683000,0.502000
ShallowConvNet,0.656000,62,0.810000,0.812000,0.537000,0.640000,0.592000,0.616000,0.479000,0.828000,0.646000,0.594000,0.627000,0.575000,0.545000,0.599000,0.618000,0.660000,0.531000,0.491000,0.792000,0.679000,0.672000,0.418000,0.696000,0.804000,0.527000,0.980000,0.595000,0.918000,0.841000,0.926000,0.509000,0.622000,0.542000,0.615000,0.699000,0.580000,0.870000,0.886000,0.731000,0.620000,0.491000,0.451000,0.609000,0.557000,0.576000,0.776000,0.604000,0.494000,0.666000,0.994000,0.526000,0.769000,0.547000,0.601000,0.554000,0.697000,0.796000,0.638000,0.674000,0.649000,0.648000,0.657000
VAR+shLDA,0.649000,62,0.591000,0.809000,0.581000,0.767000,0.522000,0.756000,0.531000,0.758000,0.736000,0.604000,0.462000,0.544000,0.597000,0.572000,0.477000,0.670000,0.546000,0.439000,0.835000,0.773000,0.574000,0.441000,0.676000,0.783000,0.656000,0.818000,0.629000,0.921000,0.662000,0.826000,0.336000,0.533000,0.708000,0.617000,0.726000,0.613000,0.857000,0.929000,0.583000,0.636000,0.617000,0.484000,0.664000,0.556000,0.620000,0.655000,0.677000,0.546000,0.710000,0.927000,0.516000,0.720000,0.716000,0.687000,0.512000,0.602000,0.840000,0.511000,0.816000,0.478000,0.623000,0.660000
Log-bandpower+shLDA,0.644000,62,0.649000,0.746000,0.576000,0.712000,0.633000,0.709000,0.485000,0.816000,0.679000,0.648000,0.492000,0.492000,0.451000,0.529000,0.552000,0.596000,0.663000,0.448000,0.827000,0.804000,0.645000,0.509000,0.630000,0.818000,0.534000,0.978000,0.574000,0.879000,0.598000,0.871000,0.473000,0.539000,0.603000,0.609000,0.769000,0.595000,0.850000,0.765000,0.564000,0.702000,0.571000,0.493000,0.559000,0.561000,0.574000,0.707000,0.584000,0.542000,0.624000,0.958000,0.553000,0.748000,0.636000,0.667000,0.521000,0.715000,0.665000,0.560000,0.843000,0.532000,0

,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20,sub-21,sub-22,sub-23,sub-24,sub-25,sub-26,sub-27,sub-28,sub-29,sub-30,sub-31,sub-32,sub-33,sub-34,sub-35,sub-36,sub-37,sub-38,sub-39,sub-40,sub-41,sub-42,sub-43,sub-44,sub-45,sub-46,sub-47,sub-48,sub-49,sub-50,sub-51,sub-52,sub-53,sub-54,sub-55,sub-56,sub-57,sub-58,sub-59,sub-60,sub-61,sub-62
method,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
EEGNet+TempDens,0.606000,62,0.610000,0.871000,0.608000,0.555000,0.692000,0.549000,0.414000,0.627000,0.464000,0.493000,0.610000,0.710000,0.715000,0.876000,0.570000,0.526000,0.753000,0.557000,0.485000,0.644000,0.567000,0.459000,0.853000,0.535000,0.635000,0.563000,0.491000,0.820000,0.616000,0.819000,0.672000,0.386000,0.543000,0.407000,0.440000,0.765000,0.792000,0.806000,0.697000,0.319000,0.608000,0.874000,0.472000,0.698000,0.717000,0.765000,0.377000,0.355000,0.576000,0.839000,0.396000,0.465000,0.642000,0.814000,0.521000,0.455000,0.746000,0.430000,0.531000,0.714000,0.521000,0.518000
FBCSP+IsolationForest,0.573000,62,0.660000,0.889000,0.566000,0.465000,0.500000,0.533000,0.418000,0.682000,0.466000,0.535000,0.474000,0.359000,0.524000,0.932000,0.499000,0.420000,0.660000,0.573000,0.628000,0.424000,0.587000,0.437000,0.863000,0.465000,0.559000,0.701000,0.621000,0.844000,0.734000,0.810000,0.763000,0.270000,0.450000,0.167000,0.644000,0.732000,0.871000,0.864000,0.503000,0.279000,0.435000,0.864000,0.332000,0.662000,0.798000,0.746000,0.399000,0.698000,0.643000,0.611000,0.408000,0.421000,0.723000,0.693000,0.500000,0.529000,0.573000,0.492000,0.347000,0.348000,0.530000,0.379000
ACM+TS+OCSVM,0.556000,62,0.667000,0.614000,0.540000,0.415000,0.539000,0.461000,0.174000,0.646000,0.723000,0.424000,0.658000,0.376000,0.475000,0.958000,0.441000,0.702000,0.752000,0.572000,0.572000,0.488000,0.415000,0.463000,0.928000,0.394000,0.488000,0.603000,0.318000,0.786000,0.631000,0.668000,0.507000,0.432000,0.593000,0.200000,0.425000,0.692000,0.820000,0.678000,0.603000,0.273000,0.625000,0.914000,0.618000,0.524000,0.543000,0.532000,0.538000,0.521000,0.514000,0.549000,0.347000,0.532000,0.414000,0.900000,0.437000,0.562000,0.675000,0.503000,0.582000,0.692000,0.461000,0.389000
VAR+Mahalanobis,0.539000,62,0.567000,0.467000,0.566000,0.464000,0.376000,0.383000,0.475000,0.762000,0.578000,0.453000,0.597000,0.352000,0.507000,0.952000,0.415000,0.589000,0.770000,0.659000,0.609000,0.552000,0.425000,0.412000,0.839000,0.543000,0.498000,0.408000,0.420000,0.861000,0.612000,0.504000,0.564000,0.434000,0.529000,0.356000,0.444000,0.671000,0.717000,0.523000,0.713000,0.328000,0.568000,0.739000,0.525000,0.543000,0.565000,0.614000,0.597000,0.477000,0.507000,0.452000,0.266000,0.497000,0.442000,0.823000,0.524000,0.663000,0.562000,0.459000,0.173000,0.642000,0.478000,0.418000
ShallowConvNet+kNN,0.535000,62,0.490000,0.784000,0.552000,0.538000,0.707000,0.569000,0.383000,0.646000,0.515000,0.514000,0.455000,0.569000,0.584000,0.537000,0.516000,0.408000,0.249000,0.480000,0.477000,0.585000,0.420000,0.360000,0.253000,0.505000,0.488000,0.790000,0.581000,0.399000,0.598000,0.732000,0.601000,0.346000,0.454000,0.457000,0.467000,0.719000,0.239000,0.617000,0.690000,0.549000,0.599000,0.770000,0.407000,0.792000,0.692000,0.379000,0.447000,0.322000,0.631000,0.787000,0.489000,0.435000,0.681000,0.477000,0.653000,0.522000,0.569000,0.433000,0.505000,0.800000,0.409000,0.520000
Log-bandpower+Mahalanobis,0.528000,62,0.609000,0.683000,0.484000,0.371000,0.549000,0.480000,0.434000,0.604000,0.428000,0.498000,0.539000,0.246000,0.461000,0.895000,0.494000,0.462000,0.611000,0.515000,0.555000,0.461000,0.508000,0.497000,0.801000,0.425000,0.620000,0.326000,0.450000,0.783000,0.691000,0.580000,0.567000,0.330000,0.486000,0.339000,0.555000,0.593000,0.621000,0.673000,0.490000,0.272000,0.549000,0.790000,0.528000,0.519000,0.448000,0.533000,0.535000,0.668000,0.452000,0.527000,0.374000,0.536000,0.436000,0.878000,0.493000,0.561000,0.48

In [6]:
display_summary_by_dataset(
    experiment,
    MI_METHODS,
    OOD_METHODS,
    datasets=datasets,
    experiment_specs=[
        ("MI BA", "mi", "MI_BA"),
        ("MI AUROC", "mi", "MI_AUROC"),
        ("OOD", "ood", "OOD_AUROC"),
    ],
)
